# Лабораторная работа №5: векторный вход и бинарная классификация

**Выполнил:** Тоц Леонид Александрович<br>
**Группа:** ИВТ-2<br>
**Дисциплина:** Базовая математика для искусственного интеллекта<br>
**Дата:** 15.10.2026

Цель: построить классификаторы OR, AND и XOR на скалярном `Value`, проверить
градиенты и объяснить роль скрытого слоя. Все вычисления используют только
стандартную библиотеку Python. Ноутбук самостоятельный: запустите все ячейки
сверху вниз в Python 3.10+ или Google Colab; другие файлы и загрузки не нужны.

Сохранены порядок и идентификаторы ячеек выданной student-заготовки.
Реализация следует методичке Dmitry Vlasov (2026); исходные `lab_guide.pdf`
и `theory.pdf` находятся рядом без изменений. Пояснения, численные проверки,
сводка и ответы на вопросы включены в эту выполненную работу.

In [1]:
# VECTOR_BINARY_CLASSIFICATION_ESSENTIALS_CELL: imports_and_helper
import math

def assert_close(actual, expected, tolerance=1e-9):
    difference = abs(actual - expected)
    assert difference <= tolerance, (
        f"Ожидалось {expected}, получено {actual}, "
        f"абсолютная ошибка {difference}"
    )


## Подготовка: скалярный граф и вектор признаков

Полный `Value` сохранён из предыдущей лабораторной. Он хранит одно число,
его градиент, родителей и локальное правило обратного прохода.
Вектор признаков остаётся списком обычных чисел и раскрывается в скалярный DAG.

In [2]:
# VECTOR_BINARY_CLASSIFICATION_ESSENTIALS_CELL: value
class Value:
    def __init__(self, data, parents=(), op=""):
        self.data = float(data)
        self.grad = 0.0
        self.parents = parents
        self.op = op
        self._backward = lambda: None

    def __repr__(self):
        return (
            f"Value(data={self.data:.6g}, grad={self.grad:.6g}, "
            f"op={self.op!r})"
        )

    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        result = Value(
            self.data + other.data,
            parents=(self, other),
            op="+",
        )

        def backward():
            self.grad += result.grad
            other.grad += result.grad

        result._backward = backward
        return result

    __radd__ = __add__

    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        result = Value(
            self.data * other.data,
            parents=(self, other),
            op="*",
        )

        def backward():
            self.grad += result.grad * other.data
            other.grad += result.grad * self.data

        result._backward = backward
        return result

    __rmul__ = __mul__

    def __neg__(self):
        result = Value(-self.data, parents=(self,), op="neg")

        def backward():
            self.grad += result.grad * (-1)

        result._backward = backward
        return result

    def __sub__(self, other):
        return self + (-other)

    def __rsub__(self, other):
        return other + (-self)

    def __pow__(self, power):
        result = Value(
            self.data ** power,
            parents=(self,),
            op=f"pow({power})",
        )

        def backward():
            local_derivative = power * self.data ** (power - 1)
            self.grad += result.grad * local_derivative

        result._backward = backward
        return result

    def sin(self):
        result = Value(math.sin(self.data), parents=(self,), op="sin")

        def backward():
            self.grad += result.grad * math.cos(self.data)

        result._backward = backward
        return result

    def exp(self):
        result = Value(math.exp(self.data), parents=(self,), op="exp")

        def backward():
            self.grad += result.grad * result.data

        result._backward = backward
        return result

    def log(self):
        result = Value(math.log(self.data), parents=(self,), op="log")

        def backward():
            self.grad += result.grad * (1 / self.data)

        result._backward = backward
        return result

    def tanh(self):
        result = Value(math.tanh(self.data), parents=(self,), op="tanh")

        def backward():
            self.grad += result.grad * (1 - result.data ** 2)

        result._backward = backward
        return result

    def backward(self):
        topo = []
        visited = set()

        def build(node):
            if node not in visited:
                visited.add(node)
                for parent in node.parents:
                    build(parent)
                topo.append(node)

        build(self)

        for node in topo:
            node.grad = 0.0
        self.grad = 1.0

        for node in reversed(topo):
            node._backward()


### Проверка скалярного механизма

Для $J=a^2+a$ при $a=2$ ожидаем $J=6$ и $dJ/da=2a+1=5$.
Повторное использование листа проверяет накопление независимых вкладов через `+=`.

In [3]:
# VECTOR_BINARY_CLASSIFICATION_ESSENTIALS_CELL: scalar_graph_refresh
a = Value(2.0)
objective = a * a + a
objective.backward()
assert_close(objective.data, 6.0)
assert_close(a.grad, 5.0)
print("Скалярный Value готов: J =", objective.data, ", dJ/da =", a.grad)


Скалярный Value готов: J = 6.0 , dJ/da = 5.0


### Векторные вспомогательные функции

`affine` вычисляет $w_0\cdot1+w_1x_1+w_2x_2$: нулевой вес является смещением.
Сырые признаки единицу не содержат; её добавляет сама функция.
В формуле $o_i=\sum_{\alpha=0}^d w_\alpha\tilde x_{i\alpha}$ индекс $i$
фиксирует объект, а $\alpha$ суммируется. В Python это цикл `zip` по координатам.
Сумма средней BCE по объектам использует отдельный немой индекс $\mu$.

`sigmoid` превращает логит в вероятность, BCE измеряет отрицательный логарифм
вероятности правильной метки, а accuracy служит диагностикой при пороге 0.5.
Производные проверяются центральной разностью с шагом $10^{-6}$.

In [4]:
# VECTOR_BINARY_CLASSIFICATION_ESSENTIALS_CELL: vector_helpers
def affine(weights, features):
    # Сумма по расширенным координатам: w_0 * 1 + w_1 * x_1 + ...
    augmented_features = [1.0, *features]
    assert len(weights) == len(augmented_features), (
        "Число весов должно включать вес постоянной координаты"
    )

    total = weights[0] * augmented_features[0]
    for weight, feature in zip(weights[1:], augmented_features[1:]):
        total = total + weight * feature
    return total

def sigmoid(logit):
    return (1.0 + (-logit).exp()) ** -1

def binary_cross_entropy(probability, target):
    target = float(target)
    assert target in (0.0, 1.0), "Метка бинарной классификации должна быть 0 или 1"
    return -(
        target * probability.log()
        + (1.0 - target) * (1.0 - probability).log()
    )

def mean_binary_cross_entropy(model, inputs, targets):
    assert len(inputs) == len(targets), "Число объектов и меток должно совпадать"
    assert len(inputs) > 0, "Нужен хотя бы один объект"

    total = Value(0.0)
    for features, target in zip(inputs, targets):
        probability = model(features)
        total = total + binary_cross_entropy(probability, target)
    return total * (1.0 / len(inputs))

def predict_label(probability):
    return 1 if probability.data >= 0.5 else 0

def classification_accuracy(model, inputs, targets):
    predictions = [predict_label(model(features)) for features in inputs]
    correct = sum(
        prediction == target
        for prediction, target in zip(predictions, targets)
    )
    return correct / len(targets)

def numerical_gradient(model, parameter, inputs, targets, step=1e-6):
    original_data = parameter.data
    parameter.data = original_data + step
    loss_plus = mean_binary_cross_entropy(model, inputs, targets).data
    parameter.data = original_data - step
    loss_minus = mean_binary_cross_entropy(model, inputs, targets).data
    parameter.data = original_data
    return (loss_plus - loss_minus) / (2.0 * step)


## Этап 1. Один sigmoid-нейрон с векторным входом

Три долгоживущих `Value` задают два входных веса и смещение.
`parameters()` возвращает ссылки на эти объекты; признаки остаются данными.

In [5]:
# VECTOR_BINARY_CLASSIFICATION_ESSENTIALS_CELL: vector_neuron
class VectorBinaryNeuron:
    def __init__(self, weights):
        assert len(weights) > 1, "Нужны вес смещения и хотя бы один входной вес"
        self.weights = [Value(weight) for weight in weights]

    def __call__(self, features):
        logit = affine(self.weights, features)
        return sigmoid(logit)

    def parameters(self):
        return self.weights


### Прямой проход OR

Начальные веса: `[0.0, 0.1, -0.2]`. Проверяем размерность и вероятности до обучения.
Для нулевого входа логит равен нулю, а вероятность равна 0.5.

In [6]:
# VECTOR_BINARY_CLASSIFICATION_ESSENTIALS_CELL: or_forward
OR_INPUTS = [[0.0, 0.0], [0.0, 1.0], [1.0, 0.0], [1.0, 1.0]]
OR_TARGETS = [0, 1, 1, 1]
or_neuron = VectorBinaryNeuron(weights=[0.0, 0.1, -0.2])
or_probabilities = [or_neuron(features).data for features in OR_INPUTS]
assert_close(or_probabilities[0], 0.5)
assert_close(or_probabilities[1], 1.0 / (1.0 + math.exp(0.2)))
assert len(or_neuron.parameters()) == 3
assert all(isinstance(parameter, Value) for parameter in or_neuron.parameters())
print("Начальные вероятности OR:", or_probabilities)


Начальные вероятности OR: [0.5, 0.45016600268752216, 0.52497918747894, 0.47502081252106]


### Градиенты OR

Проверены все три веса. Для сравнения с autograd используется центральная разность;
аналитическая формула $\nabla_{w_j}L=\frac14\sum_{\mu=1}^4
(p_\mu-y_\mu)\tilde x_{\mu j}$ дополнительно проверяется в итоговой ячейке.

In [7]:
# VECTOR_BINARY_CLASSIFICATION_ESSENTIALS_CELL: or_gradients
gradient_or = VectorBinaryNeuron(weights=[0.0, 0.1, -0.2])

or_loss = mean_binary_cross_entropy(gradient_or, OR_INPUTS, OR_TARGETS)
or_loss.backward()
or_gradient_errors = []
for parameter in gradient_or.parameters():
    expected_gradient = numerical_gradient(
        gradient_or, parameter, OR_INPUTS, OR_TARGETS,
    )
    assert_close(parameter.grad, expected_gradient, tolerance=1e-5)
    or_gradient_errors.append(abs(parameter.grad - expected_gradient))
assert len(gradient_or.parameters()) == 3
assert any(abs(parameter.grad) > 1e-8 for parameter in gradient_or.parameters())

print("BCE для OR до обучения:", or_loss.data)
print("Градиенты OR:", [parameter.grad for parameter in gradient_or.parameters()])
print("Максимальная ошибка центральной разности:", max(or_gradient_errors))


BCE для OR до обучения: 0.7200198425221698
Градиенты OR: [-0.2624584993281195, -0.25, -0.2687032961978545]
Максимальная ошибка центральной разности: 9.045564297593955e-11


### Обучение OR

Полный батч из четырёх объектов, 300 эпох, шаг 0.5.
Каждая эпоха создаёт свежий граф, вычисляет градиенты и только затем обновляет веса.
Критерии: BCE < 0.1 и accuracy = 1.0.

In [8]:
# VECTOR_BINARY_CLASSIFICATION_ESSENTIALS_CELL: train_or
learning_rate = 0.5
epochs = 300
trained_or = VectorBinaryNeuron(weights=[0.0, 0.1, -0.2])

or_losses = []
for epoch in range(epochs):
    loss = mean_binary_cross_entropy(trained_or, OR_INPUTS, OR_TARGETS)
    loss.backward()
    or_losses.append(loss.data)
    for parameter in trained_or.parameters():
        parameter.data -= learning_rate * parameter.grad

or_predictions = [trained_or(features).data for features in OR_INPUTS]
or_accuracy = classification_accuracy(trained_or, OR_INPUTS, OR_TARGETS)
assert or_losses[-1] < 0.1, or_losses[-1]
assert or_losses[-1] < or_losses[0]
assert or_accuracy == 1.0, or_accuracy
assert all(abs(parameter.data) > 0.1 for parameter in trained_or.parameters())

print("OR: loss в первой эпохе:", or_losses[0])
print("OR: loss в последней эпохе (до обновления):", or_losses[-1])
print("OR: вероятности после всех обновлений:", or_predictions)
print("OR: точность:", or_accuracy)


OR: loss в первой эпохе: 0.7200198425221698
OR: loss в последней эпохе (до обновления): 0.06236374801543139
OR: вероятности после всех обновлений: [0.13200433213232757, 0.9478517568657973, 0.9482967689524896, 0.9995440205950941]
OR: точность: 1.0


## Этап 2. Один скрытый нейрон и один выходной нейрон

$h_i=\tanh(u_0+u_1x_{i1}+u_2x_{i2})$, $p_i=\sigma(v_0+v_1h_i)$.
Всего пять параметров. AND линейно разделима; скрытый нейрон здесь позволяет
проследить градиент через композицию двух аффинных преобразований и `tanh`.

In [9]:
# VECTOR_BINARY_CLASSIFICATION_ESSENTIALS_CELL: one_hidden_model
class OneHiddenBinaryClassifier:
    def __init__(self, hidden_weights, output_weights):
        assert len(hidden_weights) > 1, "Нужны вес смещения и хотя бы один входной вес"
        assert len(output_weights) == 2, "Выходу нужны вес смещения и вес скрытого состояния"
        self.hidden_weights = [Value(weight) for weight in hidden_weights]
        self.output_weights = [Value(weight) for weight in output_weights]

    def hidden_state(self, features):
        hidden_logit = affine(self.hidden_weights, features)
        return hidden_logit.tanh()

    def __call__(self, features):
        hidden = self.hidden_state(features)
        output_logit = affine(self.output_weights, [hidden])
        return sigmoid(output_logit)

    def parameters(self):
        return self.hidden_weights + self.output_weights


### Градиенты AND

Проверены все пять параметров, включая скрытые веса.
При заданной инициализации градиент `hidden_weights[1]` равен нулю из-за
точной компенсации вкладов объектов, что подтверждает и центральная разность.
Это не означает обрыва графа. Путь к скрытому логиту задаётся формулой
$g_{r_i}=\frac{p_i-y_i}{n}v_1(1-h_i^2)$.

In [10]:
# VECTOR_BINARY_CLASSIFICATION_ESSENTIALS_CELL: and_gradients
AND_INPUTS = [[0.0, 0.0], [0.0, 1.0], [1.0, 0.0], [1.0, 1.0]]
AND_TARGETS = [0, 0, 0, 1]

gradient_and = OneHiddenBinaryClassifier(
    hidden_weights=[0.0, 0.1, -0.2],
    output_weights=[0.0, 0.3],
)

and_loss = mean_binary_cross_entropy(gradient_and, AND_INPUTS, AND_TARGETS)
and_loss.backward()
and_gradient_errors = []
for parameter in gradient_and.parameters():
    expected_gradient = numerical_gradient(
        gradient_and, parameter, AND_INPUTS, AND_TARGETS,
    )
    assert_close(parameter.grad, expected_gradient, tolerance=1e-5)
    and_gradient_errors.append(abs(parameter.grad - expected_gradient))
assert len(gradient_and.parameters()) == 5
assert any(abs(parameter.grad) > 1e-8 for parameter in gradient_and.parameters())
assert_close(gradient_and.hidden_weights[1].grad, 0.0)

print("BCE для AND до обучения:", and_loss.data)
print("Градиенты AND:", [parameter.grad for parameter in gradient_and.parameters()])
print("Максимальная ошибка центральной разности:", max(and_gradient_errors))


BCE для AND до обучения: 0.6933861312886811
Градиенты AND: [0.07247243881438471, 0.0, -0.0027100690521792425, 0.24630029365404474, 0.0013478006952490802]
Максимальная ошибка центральной разности: 6.138584085491061e-11


### Обучение AND

300 эпох, шаг 0.3, фиксированные начальные веса из методички.
Критерии: BCE < 0.05, accuracy = 1.0; в последней эпохе все пять градиентов ненулевые.

In [11]:
# VECTOR_BINARY_CLASSIFICATION_ESSENTIALS_CELL: train_and
learning_rate = 0.3
epochs = 300
trained_and = OneHiddenBinaryClassifier(
    hidden_weights=[0.0, 0.1, -0.2],
    output_weights=[0.0, 0.3],
)

and_losses = []
for epoch in range(epochs):
    loss = mean_binary_cross_entropy(trained_and, AND_INPUTS, AND_TARGETS)
    loss.backward()
    and_losses.append(loss.data)
    for parameter in trained_and.parameters():
        parameter.data -= learning_rate * parameter.grad

and_predictions = [trained_and(features).data for features in AND_INPUTS]
and_accuracy = classification_accuracy(trained_and, AND_INPUTS, AND_TARGETS)
assert and_losses[-1] < 0.05, and_losses[-1]
assert and_losses[-1] < and_losses[0]
assert and_accuracy == 1.0, and_accuracy
assert all(abs(parameter.grad) > 1e-8 for parameter in trained_and.parameters())

print("AND: loss в первой эпохе:", and_losses[0])
print("AND: loss в последней эпохе (до обновления):", and_losses[-1])
print("AND: вероятности после всех обновлений:", and_predictions)
print("AND: точность:", and_accuracy)


AND: loss в первой эпохе: 0.6933861312886811
AND: loss в последней эпохе (до обновления): 0.022928442324803684
AND: вероятности после всех обновлений: [0.0037193584223685116, 0.019203018766091172, 0.019204423277084767, 0.9524535868989039]
AND: точность: 1.0


### Зачем в маршруте нужен XOR

Один скрытый `tanh` и монотонный sigmoid-выход задают одну линейную границу
либо постоянный прогноз. XOR такой границей не разделяется.
Доказательство приведено в ответе на вопрос 7.

## Этап 3. Скрытый слой из двух нейронов и XOR

Два скрытых нейрона создают разные нелинейные характеристики. Выходной
sigmoid-нейрон объединяет их. Всего девять весов: 2 × 3 скрытых и 3 выходных.
Класс допускает произвольное положительное число скрытых нейронов;
в обязательном эксперименте используются ровно два.

In [12]:
# VECTOR_BINARY_CLASSIFICATION_ESSENTIALS_CELL: xor_model
class HiddenLayerBinaryClassifier:
    def __init__(self, hidden_layer_weights, output_weights):
        assert len(hidden_layer_weights) > 0, "Нужен хотя бы один скрытый нейрон"
        feature_count = len(hidden_layer_weights[0])
        assert feature_count > 1, "У скрытого нейрона нужны смещение и хотя бы один вход"
        assert all(
            len(weights) == feature_count
            for weights in hidden_layer_weights
        ), "Все скрытые нейроны должны иметь одинаковое число входов"
        assert len(output_weights) == len(hidden_layer_weights) + 1, (
            "Выходу нужны вес смещения и веса всех скрытых состояний"
        )
        self.hidden_weights = [
            [Value(weight) for weight in weights]
            for weights in hidden_layer_weights
        ]
        self.output_weights = [Value(weight) for weight in output_weights]

    def hidden_states(self, features):
        return [
            affine(weights, features).tanh()
            for weights in self.hidden_weights
        ]

    def __call__(self, features):
        hidden = self.hidden_states(features)
        output_logit = affine(self.output_weights, hidden)
        return sigmoid(output_logit)

    def parameters(self):
        parameters = []
        for weights in self.hidden_weights:
            parameters.extend(weights)
        parameters.extend(self.output_weights)
        return parameters


### Градиенты XOR

Центральной разностью проверены все девять параметров до обучения.
Два пути от скрытых состояний к выходу входят в единый скалярный DAG.

In [13]:
# VECTOR_BINARY_CLASSIFICATION_ESSENTIALS_CELL: xor_gradients
XOR_INPUTS = [[0.0, 0.0], [0.0, 1.0], [1.0, 0.0], [1.0, 1.0]]
XOR_TARGETS = [0, 1, 1, 0]

gradient_xor = HiddenLayerBinaryClassifier(
    hidden_layer_weights=[[0.1, 0.4, -0.3], [-0.1, -0.2, 0.5]],
    output_weights=[0.0, 0.3, -0.4],
)

xor_loss = mean_binary_cross_entropy(gradient_xor, XOR_INPUTS, XOR_TARGETS)
xor_loss.backward()
xor_gradient_errors = []
for parameter in gradient_xor.parameters():
    expected_gradient = numerical_gradient(
        gradient_xor, parameter, XOR_INPUTS, XOR_TARGETS,
    )
    assert_close(parameter.grad, expected_gradient, tolerance=1e-5)
    xor_gradient_errors.append(abs(parameter.grad - expected_gradient))
assert len(gradient_xor.parameters()) == 9
assert any(abs(parameter.grad) > 1e-8 for parameter in gradient_xor.parameters())

print("BCE для XOR до обучения:", xor_loss.data)
print("Градиенты XOR:", [parameter.grad for parameter in gradient_xor.parameters()])
print("Максимальная ошибка центральной разности:", max(xor_gradient_errors))


BCE для XOR до обучения: 0.6974895138776747
Градиенты XOR: [0.008525444612619417, 0.009933955272260968, -0.004147157354024503, -0.011573946274921518, -0.007627296378057938, -0.00029512097101965645, 0.005836030048295041, 0.014154145503698236, -0.009161276776075637]
Максимальная ошибка центральной разности: 1.232095883652029e-10


### Обучение XOR

500 эпох, шаг 0.5. Сохранена заданная асимметричная инициализация:
скрытые веса `[[0.1, 0.4, -0.3], [-0.1, -0.2, 0.5]]`, выходные `[0.0, 0.3, -0.4]`.
Критерии: BCE < 0.03, accuracy = 1.0.

In [14]:
# VECTOR_BINARY_CLASSIFICATION_ESSENTIALS_CELL: train_xor
learning_rate = 0.5
epochs = 500
trained_xor = HiddenLayerBinaryClassifier(
    hidden_layer_weights=[[0.1, 0.4, -0.3], [-0.1, -0.2, 0.5]],
    output_weights=[0.0, 0.3, -0.4],
)

xor_losses = []
for epoch in range(epochs):
    loss = mean_binary_cross_entropy(trained_xor, XOR_INPUTS, XOR_TARGETS)
    loss.backward()
    xor_losses.append(loss.data)
    for parameter in trained_xor.parameters():
        parameter.data -= learning_rate * parameter.grad

xor_predictions = [trained_xor(features).data for features in XOR_INPUTS]
xor_accuracy = classification_accuracy(trained_xor, XOR_INPUTS, XOR_TARGETS)
assert xor_losses[-1] < 0.03, xor_losses[-1]
assert xor_losses[-1] < xor_losses[0]
assert xor_accuracy == 1.0, xor_accuracy
assert all(abs(parameter.data) > 0.1 for parameter in trained_xor.parameters())

print("XOR: loss в первой эпохе:", xor_losses[0])
print("XOR: loss в последней эпохе (до обновления):", xor_losses[-1])
print("XOR: вероятности после всех обновлений:", xor_predictions)
print("XOR: точность:", xor_accuracy)


XOR: loss в первой эпохе:

 0.6974895138776747
XOR: loss в последней эпохе (до обновления): 0.025122785704812224
XOR: вероятности после всех обновлений: [0.01889303602312178, 0.9676336766552224, 0.9684286235905822, 0.01592091709544799]
XOR: точность: 1.0


## Диагностика и интерпретация

Все обязательные проверки пройдены. Порог 0.5 применяется после sigmoid.
Градиенты обнуляются внутри `backward()` при каждом новом обратном проходе.
Веса сохраняют идентичность между эпохами, промежуточные DAG создаются заново.
Результаты относятся к четырём точкам таблиц истинности: отдельного test здесь нет.

## Чек-лист перед сдачей

- [x] Все обязательные `assert` пройдены.
- [x] Центральная разность совпала с autograd для OR, AND и XOR (3 + 5 + 9 весов).
- [x] Объяснено, почему признаки не входят в `parameters()`.
- [x] Показано использование одного параметра всеми объектами выборки.
- [x] Получены требуемые BCE и accuracy для трёх моделей.
- [x] Объяснено ограничение одного скрытого нейрона на XOR.
- [x] Даны письменные ответы на все семь вопросов.


## Ответы на вопросы для самопроверки

**1. Какие вершины создаёт `affine` для двух признаков?**

Даны три общих листа $w_0,w_1,w_2$ и константные листья $1,x_1,x_2$.
Возникают три узла умножения $q_0=w_0\cdot1$, $q_1=w_1x_1$, $q_2=w_2x_2$,
затем два сложения: $s=q_0+q_1$, $o=s+q_2$.
Путь смещения существует в DAG как настоящее умножение на единицу,
поэтому $\partial o/\partial w_0=1$ получается общим правилом.
Каждый вызов создаёт новые константы и промежуточные узлы, используя прежние веса.

**2. Почему одна ссылка на `weights[0]` даёт сумму вкладов по выборке?**

Все четыре прямых прохода ссылаются на один и тот же объект веса.
Средняя BCE объединяет их потери в один DAG. Каждый потомок добавляет свой вклад
через `+=`, и вес получает
$\partial L/\partial w_j=\frac14\sum_{\mu=1}^4(p_\mu-y_\mu)\tilde x_{\mu j}$.
Деление на 4 выполняется один раз при вычислении средней потери.
`features` и метки являются фиксированными данными; они не входят в `parameters()`.
Даже если внутренние константные листья получают `grad`, цикл обучения обновляет
только возвращённые моделью веса.

**3. Какой узел хранит вероятность класса 1?**

Выходной `sigmoid`: $p=\sigma(o)\in(0,1)$. Скрытый `tanh` хранит характеристику
$h\in(-1,1)$ и не является вероятностью. Метка равна 1 при $p\ge0.5$, иначе 0.

**4. Почему BCE вместе с sigmoid даёт градиент $p-y$?**

$$\frac{\partial\ell}{\partial p}
=-\frac y p+\frac{1-y}{1-p}=\frac{p-y}{p(1-p)},\qquad
\frac{\partial p}{\partial o}=p(1-p).$$

По цепному правилу множители сокращаются:
$\partial\ell/\partial o=p-y$. Для средней потери вклад одного объекта
равен $(p_i-y_i)/n$. В коде результат получается обратным обходом скалярных
операций, а не подстановкой этой формулы. Accuracy зависит от пороговых меток,
поэтому почти всюду локально постоянна и не даёт полезного градиента.

**5. Что нарушится при обновлении `parameter.data` до `loss.backward()`?**

Сохранённые результаты прямого прохода относятся к старым весам,
а замыкания некоторых операций читают текущее `data` родителей.
Возникнет смесь состояний, и градиент не будет производной построенной потери.
Правильный порядок: свежий DAG → `backward()` → обновление всех параметров.

**6. Почему одинаковые скрытые нейроны не приобретают разные роли?**

При полной симметрии, то есть одинаковых скрытых весах **и одинаковых весах
соединения с выходом**, оба нейрона дают одинаковые состояния и градиенты.
Детерминированный градиентный спуск сохраняет это равенство, поэтому два нейрона
работают как один. Важно учитывать выходные связи:
$g_{r_{ih}}=g_{o_i}v_h(1-h_{ih}^2)$; при разных $v_h$ одинаковые скрытые веса
сами по себе не гарантируют одинаковых градиентов. В фиксированном XOR-опыте
различаются и начальные скрытые веса, и выходные связи.

**7. Почему XOR является тестом выразительности?**

OR проверяет обучение одной линейной границы, AND со скрытым нейроном проверяет
цепное правило, а XOR требует нескольких нелинейных характеристик.
Для одного скрытого нейрона $p=\sigma(v_0+v_1\tanh(u_0+u_1x_1+u_2x_2))$.
Монотонность функций превращает порог $p\ge0.5$ в полуплоскость или постоянный
ответ. Ни одна полуплоскость не разделяет XOR. Действительно, для любого
$f(x)=c_0+c_1x_1+c_2x_2$:

$$f(0,1)+f(1,0)=f(0,0)+f(1,1).$$

Для положительных точек левая сумма должна быть неотрицательной, а для
отрицательных правая должна быть строго отрицательной, что невозможно.
Два скрытых `tanh` создают две разные характеристики, и выход комбинирует их.
Четыре логические точки позволяют проверить эту возможность; они не оценивают
качество прикладного классификатора на новых данных.

## Связь с MLP

Один скрытый слой можно записать компактно:
$r_i=U\tilde x_i$, $h_i=\tanh(r_i)$, $p_i=\sigma(v^T\tilde h_i)$.
В XOR $U$ имеет размер $2\times3$, а $v$ содержит три веса с учётом смещения.
В данном коде каждая такая формула раскрыта в скалярные операции. Для MLP
аффинное преобразование и нелинейность повторяются по слоям; обратный проход
по-прежнему использует цепное правило и суммирование вкладов в общих параметрах.

In [15]:
print("Проверка аналитического градиента OR и общих параметров:")
for j, parameter in enumerate(gradient_or.parameters()):
    expected = sum(
        (gradient_or(x).data - y) * [1.0, *x][j]
        for x, y in zip(OR_INPUTS, OR_TARGETS)
    ) / len(OR_INPUTS)
    assert_close(parameter.grad, expected)
    assert parameter is gradient_or.parameters()[j]
    print(f"w{j}: autograd={parameter.grad:.12f}, аналитически={expected:.12f}")

print("\nИтог после всех обновлений (свежий прямой проход):")
print(f"{'Задача':<8} {'BCE':>12} {'Accuracy':>10} {'Верные метки':>18}")
for task, model, inputs, targets, limit in [
    ('OR', trained_or, OR_INPUTS, OR_TARGETS, 0.1),
    ('AND', trained_and, AND_INPUTS, AND_TARGETS, 0.05),
    ('XOR', trained_xor, XOR_INPUTS, XOR_TARGETS, 0.03),
]:
    final_loss = mean_binary_cross_entropy(model, inputs, targets).data
    labels = [predict_label(model(x)) for x in inputs]
    accuracy = classification_accuracy(model, inputs, targets)
    assert final_loss < limit and accuracy == 1.0 and labels == targets
    print(f"{task:<8} {final_loss:>12.9f} {accuracy:>10.1f} {str(labels):>18}")
    print("  Веса:", [round(p.data, 9) for p in model.parameters()])
print("OK: все этапы и аналитическая проверка выполнены")

Проверка аналитического градиента OR и общих параметров:
w0: autograd=-0.262458499328, аналитически=-0.262458499328
w1: autograd=-0.250000000000, аналитически=-0.250000000000
w2: autograd=-0.268703296198, аналитически=-0.268703296198

Итог после всех обновлений (свежий прямой проход):
Задача            BCE   Accuracy       Верные метки
OR        0.062167395        1.0       [0, 1, 1, 1]
  Веса: [-1.883351982, 4.792499208, 4.783459606]
AND       0.022805302        1.0       [0, 0, 0, 1]
  Веса: [-2.672670146, 1.912007141, 1.911980455, -0.887490337, 4.748072114]
XOR       0.025026250        1.0       [0, 1, 1, 0]
  Веса: [1.347017495, 2.974259388, -2.837894449, 1.544078643, -3.172591026, 3.245772954, 3.779004017, -4.34634711, -4.308728159]
OK: все этапы и аналитическая проверка выполнены


## Итоговые результаты

| Задача | Параметров | Эпох | Шаг | Начальная BCE | Итоговая BCE | Accuracy |
|---|---:|---:|---:|---:|---:|---:|
| OR | 3 | 300 | 0.5 | 0.720019843 | 0.062167395 | 1.0 |
| AND | 5 | 300 | 0.3 | 0.693386131 | 0.022805302 | 1.0 |
| XOR | 9 | 500 | 0.5 | 0.697489514 | 0.025026250 | 1.0 |

Итоговая BCE вычислена после последнего обновления. Последний элемент истории
содержит BCE перед этим обновлением, как в выданном цикле обучения.

Один sigmoid-нейрон выучил OR. AND подтвердил прохождение градиента через
скрытый `tanh`. Два разных скрытых нейрона позволили решить XOR.
Механизм `Value` остался скалярным на всех этапах.
Все результаты воспроизводятся без случайной инициализации и внешних датасетов.